# Qwen3-4B on Spider dev: RQ1 and RQ2

This notebook reproduces every table and figure in the report (`report/RQ1.tex`, `report/RQ2.tex`) on a local machine.
It runs the six variants below over the 1034 Spider dev questions, then rebuilds each table and figure from those runs.
Every step calls the code in `agent_ext/`, so the results match runs made from the command line.

| variant | few-shot RAG (5 nearest train examples) | ReAct loop | act tools | Report Name |
|---|---|---|---|---|
| `base` | – | – | – (the original A1 prompt) | System A |
| `rag` | ✓ | – | – | System A + Few-Shot RAG |
| `react` | – | ✓ | `execute` | System A + ReAct Loop + Execute Tool |
| `react_parse` | – | ✓ | `execute` + `spider_parse` | System A + ReAct Loop + Spider Parse Tool |
| `rag_react` | ✓ | ✓ | `execute` | System A + Few-Shot RAG + ReAct Loop + Execute Tool |
| `rag_react_parse` | ✓ | ✓ | `execute` + `spider_parse` | System B (Few-Shot RAG + ReAct Loop + Execute Tool + Spider Parse Tool)|

| Report | Section below |
|---|---|
| Table `tab:rq1-summary` (RQ1 summary) | 3.1 |
| Tables `tab:acc`, `tab:parse`, `tab:comp`, `tab:exec`, `tab:exec-diff`, `tab:eff`, `tab:agree` | 3.2–3.8 |
| Tables `tab:abl-em`, `tab:abl-ex`, `tab:abl-comp` (ablation) | 4.1–4.3 |
| Table `tab:fb` (questions fixed vs. broken per component) | 4.4 |
| Table `tab:desc` (System B trajectory features) | 4.5 |
| Figures `fig:traj-repair`, `fig:traj-distance` (`report/traj_repair.png`, `report/traj_distance.png`) | 4.6 |

`report/loop.png` (Figure `fig:react-loop`) is a hand-drawn diagram of the loop, not data, so it is not regenerated.

## 1. Configuration

Use the kernel described in the README (`.venv`, with the schema-only Spider databases, NLTK data,
Qwen3-4B and all-MiniLM-L6-v2 already downloaded to `a1_lib/models`).

- `RUN_ID`: every variant writes to `agent_ext/results/<variant>/<RUN_ID>/`, and every table below reads from there.
  Use a fresh name so earlier results are not overwritten.
- `GENERATE`: `True` runs the model for all six variants (needs a GPU).
  `False` skips generation and rebuilds the tables and figures from the existing results of `RUN_ID`.
- `LIMIT`: `None` for the full dev set (1034 questions, as in the report), or a small number for a quick check.

Greedy decoding is deterministic per batch, but padding in a bf16 batch can change a few outputs, so a rerun can differ
from the report by a few questions (a few tenths of a point).

In [ ]:
import os, sys
from pathlib import Path
from IPython.display import Image, Markdown, display

# the repo root is the first directory upward that holds agent_ext/; importing agent_ext then puts a1_lib and
# spider on sys.path and sets NLTK_DATA
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "agent_ext").is_dir())))
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
from agent_ext import REPO, RESULTS_DIR
from agent_ext import run as ext_run, report_tables as rt
os.chdir(REPO)

RUN_ID = "repro"      # every variant writes to agent_ext/results/<variant>/<RUN_ID>/
GENERATE = True       # False: skip the model runs and rebuild tables/figures from existing RUN_ID results
LIMIT = None          # None = full dev (1034); a small number (e.g. 20) for a quick check
N_TRACES = 5          # random traces saved per variant, from successes and from failures each
SEED = 0
BATCH_SIZE = 64       # questions generated together; halved automatically on CUDA out-of-memory
MODELS_PATH = REPO / "a1_lib/models"
VARIANTS = list(ext_run.VARIANTS)   # base, rag, react, react_parse, rag_react, rag_react_parse

print("repo:", REPO)

In [ ]:
def run_main(main, args):
    # call a script's main() in this process with the given command-line args
    old = sys.argv
    sys.argv = ["agent_ext", *args]
    try:
        main()
    finally:
        sys.argv = old

def show(title, df):
    display(Markdown(f"**{title}**"))
    display(df)

## 2. Run the six variants

Each variant is one full run of Qwen3-4B (bf16, greedy decoding, thinking off) followed by Spider's official evaluator.
Rerunning with the same `RUN_ID` overwrites that run; add `"--resume"` to `variant_args` to continue an interrupted one.

In [ ]:
def variant_args(variant, extra=()):
    a = ["--variant", variant, "--run-id", RUN_ID, "--seed", str(SEED), "--save-traces", str(N_TRACES),
         "--batch-size", str(BATCH_SIZE), "--local-only", "--models-path", str(MODELS_PATH), *extra]
    return a + (["--limit", str(LIMIT)] if LIMIT else [])

if GENERATE:
    for variant in VARIANTS:
        print(f"=== {variant}", flush=True)
        run_main(ext_run.main, variant_args(variant))

missing = [v for v in VARIANTS if not (RESULTS_DIR / v / RUN_ID / "metrics.json").exists()]
assert not missing, f"no finished run {RUN_ID!r} for {missing}; set GENERATE = True or pick another RUN_ID"

## 3. RQ1: System A vs. System B

System A is `base` and System B is `rag_react_parse`.

### 3.1 Summary (Table `tab:rq1-summary`)

Change is B − A in points, except for the cost rows, where it is B / A. It collects the headline rows of the tables in 3.2–3.8.

In [ ]:
show("Summary of System A vs System B", rt.summary(RUN_ID, MODELS_PATH))

### 3.2 Exact match by difficulty (Table `tab:acc`)

In [ ]:
show("Exact match (%) by difficulty", rt.accuracy(RUN_ID))

### 3.3 Output validity (Table `tab:parse`)

In [ ]:
show("Output validity", rt.validity(RUN_ID))

### 3.4 Component matching (Table `tab:comp`)

In [ ]:
show("Component matching (%)", rt.components(RUN_ID))

### 3.5 Exact match and execution accuracy (Table `tab:exec`)

Execution is against schema-only databases (all tables empty), so it mostly measures whether a query runs.

In [ ]:
show("Whole-query metrics (%)", rt.whole_query(RUN_ID))

### 3.6 Execution accuracy by difficulty (Table `tab:exec-diff`)

In [ ]:
show("Execution accuracy (%) by difficulty (schema-only databases)", rt.execution_accuracy(RUN_ID))

### 3.7 Efficiency (Table `tab:eff`)

Input tokens are recomputed from the saved prompts and traces with the Qwen3 tokenizer (CPU only). Wall-clock time is the generation time of the run divided by the number of questions, so it depends on the GPU.

In [ ]:
show("Mean cost per question", rt.efficiency(RUN_ID, MODELS_PATH))

### 3.8 Agreement (Table `tab:agree`)

In [ ]:
show("Per-question agreement on exact match", rt.agreement(RUN_ID))

## 4. RQ2: ablation and trajectory analysis

### 4.1 Exact match by variant (Table `tab:abl-em`)

In [ ]:
show("Exact-match accuracy (%) by difficulty", rt.ablation(RUN_ID))

### 4.2 Execution accuracy by variant (Table `tab:abl-ex`)

In [ ]:
show("Execution accuracy (%) by difficulty (schema-only databases)", rt.ablation_execution(RUN_ID))

### 4.3 Component matching by variant (Table `tab:abl-comp`)

P = precision (Spider's "accuracy"), R = recall, F1, over all questions.

In [ ]:
show("Component-matching scores (%)", rt.ablation_components(RUN_ID))

### 4.4 Questions fixed vs. broken per component (Table `tab:fb`)

Each component is added to System A to form the variant, which is compared against System B minus that component
(the parse tool is also compared against the execute-only loop). *Fixed*: exact match in the variant but not in the
comparison; *Broke*: the reverse.

In [ ]:
show("Per-component comparison", rt.fixed_broke(RUN_ID))

### 4.5 System B trajectory features (Table `tab:desc`)

Per difficulty and exact-match correctness: how many trajectories hit a tool error and how many of those were repaired,
and how the final SQL relates to the first draft.

In [ ]:
show("System B trajectories", rt.trajectory_stats(RUN_ID))

### 4.6 Figures `fig:traj-repair` and `fig:traj-distance`

Writes `report/traj_repair.png` and `report/traj_distance.png`, the files the report includes, and prints the numbers behind them. "unchanged" is the share of trajectories whose final query is their first parsable draft (distance 0).

In [ ]:
from agent_ext import plot_features
run_main(plot_features.main, ["--run-id", RUN_ID, "--models-path", str(MODELS_PATH)])
for name in (plot_features.REPAIR_PNG, plot_features.DISTANCE_PNG):
    display(Image(filename=str(plot_features.OUT_DIR / name)))